# Audio Generation

This notebook demonstrates how to generate audio using:
1. [Coqui TTS](https://github.com/coqui-ai/TTS)
2. [MusicGen](https://github.com/facebookresearch/audiocraft/blob/main/docs/MUSICGEN.md)

## Table of Contents
1. [Intro to TTS](#intro)
2. [Setup Environment](#setup)
3. [Coqui TTS Inference](#coqui_inference)
4. [Coqui TTS Training](#coqui_training)
5. [MusicGen Inference](#musicgen)

---

## 1. Intro

<p align="center">
  <img src="https://www.researchgate.net/publication/378149981/figure/fig2/AS:11431281223517189@1707795653490/Structure-of-TTS-models-based-on-deep-learning-The-autoregressive-models-follows-the.png" width="800"/>
</p>

Firstly, we’ll explore how modern TTS systems convert written text into natural-sounding speech. While there are several architectures, most follow these main steps:

1. **Text Preprocessing**  
   - Normalizes text (expanding abbreviations, handling numbers, etc.).  
   - Often applies a Grapheme-to-Phoneme (G2P) conversion to produce phonetic sequences.

2. **Linguistic/Phonetic Analysis**  
   - Breaks text into smaller units (phonemes, syllables, or words) and applies prosodic features (e.g., stress or intonation markers).  
   - Captures the rhythm and melody of speech in a form suitable for acoustic modeling.

3. **Acoustic Model**  
   - A neural network (e.g., Tacotron, FastSpeech, or VITS) predicts an **acoustic representation** (commonly a mel-spectrogram) from the sequence of phonemes.  
   - In attention-based models like Tacotron, an alignment mechanism learns how each text token corresponds to frames of the mel-spectrogram.

4. **Vocoder**  
   - Transforms the acoustic representation into a raw audio waveform. Models like WaveGlow, HiFi-GAN, or LPCNet focus on producing high-fidelity speech with minimal artifacts.

Some modern TTS frameworks (e.g., **Coqui TTS**) can bundle these steps into more **end-to-end** pipelines, where much of the text analysis and spectrogram generation happen implicitly inside a single model.

## 2.1 Setup (~3 mins on Colab)

We'll install the necessary libraries for TTS and MusicGen in this section.

We'll also install `IPython.display` dependencies to play the generated audio inside the notebook.

In [ ]:
!pip install --upgrade --quiet pip
!pip install --upgrade --quiet transformers datasets[audio]
!pip install --quiet TTS

from IPython.display import Audio

In [ ]:
!sudo apt-get install espeak

## 3 Coqui TTS Inference (~5 min)

We'll load a pretrained model from [Coqui TTS](https://github.com/coqui-ai/TTS) and run a simple inference.

Coqui TTS comes with a list of pretrained models for different model types (ex: TTS, vocoder), languages, datasets used for training and architectures.

You can either use your own model or the release models under TTS.

Use `!tts --list_models` to find out the availble models.

In [ ]:
!tts --list_models

In [ ]:
!tts --text "hello world" \
--model_name "tts_models/en/ljspeech/glow-tts" \
--out_path output.wav

In [ ]:
from IPython.display import Audio

Audio("output.wav")

### 3.1 Synthesize speech using Speaker ID

- A TTS model can be either trained on a single speaker voice or multispeaker voices. This training choice is directly reflected on the inference ability and the available speaker voices that can be used to synthesize speech.

- If you want to run a multispeaker model from the released models list, you can first check the speaker ids using `--list_speaker_idx` flag and use this speaker voice to synthesize speech.

In [ ]:
# list the possible speaker IDs.
!tts --model_name "tts_models/en/vctk/vits" \
--list_speaker_idxs

In [ ]:
!tts --text "Trying out specific speaker voice"\
--out_path spkr-out.wav --model_name "tts_models/en/vctk/vits" \
--speaker_idx "p341"

In [ ]:
Audio("spkr-out.wav")

### 3.2 Synthesize using your specified speaker

If you want to use an external speaker to synthesize speech, you need to supply `--speaker_wav` flag along with an external speaker encoder path and config file.

First we need to get the speaker encoder model, its config and a referece `speaker_wav`



In [ ]:
!wget https://github.com/coqui-ai/TTS/releases/download/speaker_encoder_model/config_se.json
!wget https://github.com/coqui-ai/TTS/releases/download/speaker_encoder_model/model_se.pth.tar
!wget https://github.com/coqui-ai/TTS/raw/speaker_encoder_model/tests/data/ljspeech/wavs/LJ001-0001.wav

In [ ]:
!tts --model_name tts_models/multilingual/multi-dataset/your_tts \
--encoder_path model_se.pth.tar \
--encoder_config config_se.json \
--speaker_wav LJ001-0001.wav \
--text "Are we not allowed to dim the lights so people can see that a bit better?"\
--out_path spkr-out.wav \
--language_idx "en"

In [ ]:
Audio("spkr-out.wav")

## 4. Coqui TTS Training (2 epochs = ~30 min)

Coqui TTS provides extensive documentation for training your own models. Training typically requires a dataset of audio clips and corresponding transcripts.

### 4.1 Loading Dataset


In [ ]:
import os

from TTS.tts.configs.shared_configs import BaseDatasetConfig

output_path = "tts_train_dir"
if not os.path.exists(output_path):
    os.makedirs(output_path)

In [ ]:
# Download and extract LJSpeech dataset.

!wget -O $output_path/LJSpeech-1.1.tar.bz2 https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
!tar -xf $output_path/LJSpeech-1.1.tar.bz2 -C $output_path

In [ ]:
dataset_config = BaseDatasetConfig(
    formatter="ljspeech", meta_file_train="metadata.csv", path=os.path.join(output_path, "LJSpeech-1.1/")
)

### 4.2 Model Training

In this seminar we will be training `GlowTTS`, but CoquiTTS has many recipes under `TTS/recipes/` that provide a good starting point. For more info on `GlowTTS` refer to the following [blogpost](https://jaketae.github.io/study/glowtts/).

We will begin by initializing the model training configuration.

In [ ]:
# GlowTTSConfig: all model related values for training, validating and testing.
from TTS.tts.configs.glow_tts_config import GlowTTSConfig
config = GlowTTSConfig(
    batch_size=32,
    eval_batch_size=16,
    num_loader_workers=4,
    num_eval_loader_workers=4,
    run_eval=True,
    test_delay_epochs=-1,
    epochs=100,
    text_cleaner="phoneme_cleaners",
    use_phonemes=True,
    phoneme_language="en-us",
    phoneme_cache_path=os.path.join(output_path, "phoneme_cache"),
    print_step=25,
    print_eval=False,
    mixed_precision=True,
    output_path=output_path,
    datasets=[dataset_config],
    save_step=1000,
)

Next we will initialize the audio processor which is used for feature extraction and audio I/O.

In [ ]:
!pip install -q numpy==1.21.0

In [ ]:
from TTS.utils.audio import AudioProcessor
ap = AudioProcessor.init_from_config(config)
# Modify sample rate if for a custom audio dataset:
# ap.sample_rate = 22050

Next we will initialize the tokenizer which is used to convert text to sequences of token IDs.  If characters are not defined in the config, default characters are passed to the config.

In [ ]:
from TTS.tts.utils.text.tokenizer import TTSTokenizer
tokenizer, config = TTSTokenizer.init_from_config(config)

Next we will load data samples. Each sample is a list of ```[text, audio_file_path, speaker_name]```. You can define your custom sample loader returning the list of samples.

In [ ]:
from TTS.tts.datasets import load_tts_samples
train_samples, eval_samples = load_tts_samples(
    dataset_config,
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=config.eval_split_size,
)

Now we're ready to initialize the model.

Models take a config object and a speaker manager as input. Config defines the details of the model like the number of layers, the size of the embedding, etc. Speaker manager is used by multi-speaker models.

In [ ]:
from TTS.tts.models.glow_tts import GlowTTS
model = GlowTTS(config, ap, tokenizer, speaker_manager=None)

The training will take a long time, so as an alternative to see how things work, we can overfit on one batch (a.k.a. **one batch test**).
To do this, simply make
```train_samples = train_samples[:batch_size]```

Also, would probably need to set ```TrainingArgs```:
```
training_args = TrainingArgs(
    shuffle=False,          # Disable shuffling
    steps_per_epoch=1,      # One step per epoch
    epochs=100,             # Adjust number of epochs
    output_path="output/",
    lr=1e-4,                # Consider increasing learning rate
    save_step=100,
    save_optimizer_state=True,
)
```

In [ ]:
from trainer import Trainer, TrainerArgs
training_args = TrainerArgs(
    shuffle=False,          # Disable shuffling
    steps_per_epoch=1,      # One step per epoch
    epochs=100,             # Adjust number of epochs
    output_path="output/",
    lr=1e-4,                # Consider increasing learning rate
    save_step=100,
    save_optimizer_state=True,
)

trainer = Trainer(
    training_args, config, output_path, model=model, train_samples=train_samples[:1], eval_samples=eval_samples[:1]
)

In [ ]:
trainer.fit()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=tts_train_dir

### 4.3 Test the model

You can see from the test output that our tiny model has overfit to the data, and basically memorized this one sentence.

When you start training your own models, make sure your testing data doesn't include your training data 😅

Save the checkpoint

In [ ]:
import glob, os
output_path = "tts_train_dir"
ckpts = sorted([f for f in glob.glob(output_path+"/*/*.pth")])
configs = sorted([f for f in glob.glob(output_path+"/*/*.json")])

Infer the model. You can download the checkpoint of the already trained (for 20 epochs) model [here](). Then just paste the path to it and config below.

In [ ]:
!tts --text "This is the test script for the GlowTTS model" \
      --model_path "/content/tts_train_dir/run-February-28-2025_01+53PM-0000000/best_model_1624.pth" \
      --config_path "/content/tts_train_dir/run-February-28-2025_01+53PM-0000000/config.json" \
      --out_path out.wav

In [ ]:
from IPython.display import Audio
Audio("out.wav")

## 5. MusicGen Inference




### 5.1 MusicGen Architecture

**MusicGen** is a single-stage, auto-regressive Transformer model for **text-to-music generation**. In essence:

1. **Text Encoder**: The model encodes an input text prompt (e.g., “A relaxing guitar melody in a serene environment”).
2. **Auto-Regressive Decoder**: A Transformer predicts a sequence of **discrete audio tokens** that correspond to the compressed representation of an audio waveform.
3. **Neural Codec Decoder**: These tokens are decoded back to raw audio waveform using a learned neural audio codec (EnCodec).

<p align="center">
  <img src="https://user-images.githubusercontent.com/76463150/260439306-81c81c8d-1f9c-41d0-b881-9491766def8e.png" width="600"/>
</p>

The key novelty (or major advantage) lies in the use of **EnCodec** as the front-end for audio compression, which drastically reduces the sequence length the Transformer must generate.

In turn, the compressed representation is produced by **Residual Vector Quantization (Residual VQ)**.

This synergy allows MusicGen to operate on tokens at a much lower temporal resolution compared to raw samples, making it computationally feasible to train a large language-model-like Transformer on audio.

---

### 5.2 EnCodec: Neural Audio Codec

<p align="center">
  <img src="https://publish-01.obsidian.md/access/93d590b5e88c06a4619cd08dc2123a4d/slp/attachments/Encodec%20model.png" width="600"/>
</p>

### 5.2.1 Why use a Neural Codec and EnCodec Architecture

EnCodec follows a **multi-scale convolutional** or sometimes a **convolution + transformer** design (depending on the version) that learns an **encoder-decoder** pipeline:
1. **Encoder**: Processes the raw waveform through several downsampling layers and non-linear transformations to yield a continuous latent representation at a smaller temporal resolution.
2. **Quantizer (with Residual VQ)**: This takes the continuous latent vectors and maps them to discrete codebook entries. Specifically, multiple codebooks are used in a **residual** fashion (elaborated in Section 3 below).
3. **Decoder**: Upsamples and transforms the discrete codes back into a time-domain waveform. It generally mirrors the encoder’s structure (with transposed convolutions or equivalent blocks).

**Key result**: Instead of generating waveforms at 32 kHz (which would be 32,000 samples per second in a single channel, or more for stereo), we only need to generate a manageable sequence of discrete tokens (e.g., 4 codebooks × 50 tokens/sec = 200 tokens/sec).

---

### 5.3 Residual Vector Quantization (Residual VQ)


<p align="center">
  <img src="https://drive.google.com/uc?export=view&id=1yTv2fpQBYqAMUbIzz0XpU0v8fmZkE9ak" width="600"/>
  <figcaption style="font-style: italic; text-align: center;">VQ Process Step 1</figcaption>
</p>

<p align="center">
  <img src="https://drive.google.com/uc?export=view&id=1QdYLl7CWzcl0K20awJpFHJUd6-s53qPg" width="600"/>
  <figcaption style="font-style: italic; text-align: center;">VQ Process Step 2</figcaption>
</p>

### 5.3.1 Standard VQ vs. Residual VQ

### **Vector Quantization (VQ)**
You take a continuous latent vector $\mathbf{z}$ and find the closest entry from a learned codebook $\{ \mathbf{e}_1, \ldots, \mathbf{e}_K \}$. That index is your discrete code, and $\mathbf{e}_k$ approximates $\mathbf{z}$.

### **Residual VQ**
Instead of mapping each vector into a single codebook, you iteratively refine it across multiple codebooks. Concretely:

1. Map $\mathbf{z}^{(0)} = \mathbf{z}$ to some codebook embedding $\mathbf{e}^{(1)}$.
2. Compute a residual $\mathbf{r}^{(1)} = \mathbf{z}^{(0)} - \mathbf{e}^{(1)}$.
3. Quantize $\mathbf{r}^{(1)}$ with a second codebook to get $\mathbf{e}^{(2)}$.
4. Repeat for multiple codebooks:
   $$
     \mathbf{z}^{(l)} = \mathbf{z}^{(l-1)} - \mathbf{e}^{(l)}, \quad l = 1, 2, \ldots, L
   $$
5. Summation: The final approximation $\hat{\mathbf{z}}$ is:
   $$
     \hat{\mathbf{z}} = \sum_{i=1}^L \mathbf{e}^{(i)}
   $$

This **multi-stage** approach (i.e., multiple codebooks) achieves higher fidelity compression at the same bit rate compared to single VQ. Each codebook refines what the previous codebook missed.

---

### 5.4 Transformer-Based Generative Model

![](https://miro.medium.com/v2/resize:fit:1400/0*PGJb7N202MMcHJ_p)

### 5.4.1 Architecture Overview

**MusicGen** uses a single-stage, **auto-regressive Transformer** to predict a flattened sequence of **codebook indices** that represent compressed audio. Each time step has multiple codebooks (e.g., 4) in an interleaved token sequence. The Transformer includes:

1. **Input Embeddings** for each quantized audio token  
2. **Positional Encoding** to maintain ordering  
3. **Self-Attention + Cross-Attention** on prior tokens and text embeddings  
4. **Feed-Forward Layers** (typical Transformer MLP blocks)  

**Text Conditioning** occurs via a separate **text encoder** (e.g., BART/T5) whose embeddings are used in cross-attention, guiding the model to generate music aligned with the prompt.

Unlike multi-stage pipelines (e.g., spectrogram + vocoder), MusicGen **directly** goes from text embeddings to discrete audio tokens in one step, then decodes these tokens with **EnCodec** to obtain the waveform.

This approach reduces domain gaps and can improve end-to-end stability.

---

## 5.5 Training and Generation Procedure

### 5.5.1 Training Objective
MusicGen is trained on large-scale music data (tens of thousands of hours of licensed music). The training objective is a standard **cross-entropy** loss over the next token in the sequence:
$$
    \mathbf{L} = - \sum_{t=1}^{T} \log P(\mathbf{x}_t \mid \mathbf{x}_{< t}, \mathbf{c}),
$$
where:
- $\mathbf{x}_t$ is the discrete token (for each codebook at time $t$).
- $\mathbf{c}$ is the text prompt encoding.
- $T$ is the length of the compressed audio sequence.

### 5.5.2 Decoding / Generation

![](https://miro.medium.com/v2/resize:fit:1400/1*uBeLtV1rY-IFRuXCnWHSMg.png)

During inference, MusicGen **auto-regressively** predicts a sequence of compressed audio tokens—one “time step” of tokens per generation step, possibly interleaving multiple residual codebooks. Once the full token sequence is generated, it is fed into **EnCodec** to reconstruct the high-fidelity waveform at 32 kHz or 48 kHz.  

A key design choice is **how** to arrange tokens from multiple codebooks in the sequence. Four common patterns (illustrated in the figure) are:

1. **Flattening Pattern**: Concatenate codebook tokens in blocks—e.g., all tokens from codebook 1, then all tokens from codebook 2, etc.  
2. **Parallel Pattern**: Interleave tokens from each codebook at every step, producing one token per codebook per time step.  
3. **VALL-E Pattern**: Similar to parallel interleaving, but the codebooks are ordered in a specific sequence with certain offsets to align or delay individual codebooks.  
4. **Delay Pattern**: Each codebook’s tokens start at different time steps (offset from one another), allowing the model to refine partial reconstructions as new codebooks come online.

Regardless of the chosen pattern, the **Transformer** attends to previously generated tokens (via self-attention) and to the text prompt’s encoded representation (cross-attention). This procedure optimizes the **cross-entropy** objective during training, while at generation time it produces music aligned with the text prompt in a single-stage pipeline: from prompt → compressed tokens → final audio.

---

## 5.6 Core Idea and Modifications

1. **Large-Scale Text-to-Music**: MusicGen was trained on a large dataset of music, enabling it to handle diverse musical styles and instruments.
2. **Leverage a high-fidelity neural codec (EnCodec)** to drastically reduce sequence length.  
3. **Train a single-stage auto-regressive Transformer**: By compressing the audio to a small number of tokens per second, the Transformer can be large and powerful without being overwhelmed by extremely long sequences.
4. **Residual VQ** to yield a compact yet high-quality representation of audio.  
5. **Cross-Attention** to textual embeddings ensures the generated music aligns semantically with the prompt.  
6. **Simple, direct pipeline**: The model does not produce intermediate spectrograms or rely on a separate vocoder; instead, it directly produces the final compressed representation used by EnCodec’s decoder.

---

## 5.7 Generation


We'll load the pre-trained `facebook/musicgen` small model checkpoints from the [pre-trained weights](https://huggingface.co/models?search=facebook/musicgen-) on the Hugging Face Hub. The checkpoints for medium and large can also be found there.

In [ ]:
!pip install datasets

In [ ]:
from transformers import MusicgenForConditionalGeneration
from IPython.display import Audio
import torch
import scipy
from transformers import AutoProcessor
from datasets import load_dataset

In [ ]:
# | model types are =>      small,  medium,   melody,   large |
# | size of models are =>   300M,   1.5B,     1.5B,     3.3B  |

model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small")

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model.to(device)

### 5.8 Model Config

The model's generation config provides with default parameters that control the generation process, such as sampling, guidance scale and number of generated tokens.

In [ ]:
model.generation_config

MusicGen supports two modes: **greedy** and **sampling**. In practice, **sampling** tends to produce higher-quality, more diverse outputs. It is enabled by default and can be explicitly activated by setting `do_sample=True` in `MusicgenForConditionalGeneration.generate`.

### 5.9.1 Unconditional Generation

For “null” (unconditional) generation, retrieve inputs using `MusicgenForConditionalGeneration.get_unconditional_inputs`. This lets the model generate audio auto-regressively without any text conditioning.

In [ ]:
unconditional_inputs = model.get_unconditional_inputs(num_samples=1)

audio_values = model.generate(**unconditional_inputs, do_sample=True, max_new_tokens=256)

In [ ]:
sampling_rate = model.config.audio_encoder.sampling_rate
Audio(audio_values[0].cpu().numpy(), rate=sampling_rate)

You can also save it on disk as follows

In [ ]:
scipy.io.wavfile.write("musicgen_out.wav", rate=sampling_rate, data=audio_values[0, 0].cpu().numpy())

See number of seconds in generated audio

In [ ]:
duration = 256 / model.config.audio_encoder.frame_rate

duration

### 5.9.2 Text-Conditional Generation

The model can generate an audio sample conditioned on a text prompt through the use of the `MusicgenProcessor` to pre-process
the inputs.

In [ ]:
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")

inputs = processor(
    text=["Romantic ballad with a gentle acoustic guitar and soft vocal harmonies.", "80s synthwave track with retro synths, driving bass, and a nostalgic vibe."],
    padding=True,
    return_tensors="pt",
)

In [ ]:
audio_values = model.generate(**inputs.to(device), do_sample=True, guidance_scale=3, max_new_tokens=256)

Audio(audio_values[0].cpu().numpy(), rate=sampling_rate)

In [ ]:
Audio(audio_values[1].cpu().numpy(), rate=sampling_rate)



The `guidance_scale` holds the same role as in diffusion models - it controls the balance between conditional logits (from text prompts) and unconditional logits (from a 'null' prompt) in classifier-free guidance (CFG). A higher value makes the output align more closely with the prompt but may reduce audio quality. CFG is active when `guidance_scale > 1`. For optimal results, use the default `guidance_scale=3` for text- and audio-conditional generation.


### 5.9.3 Audio-Prompted Generation

The same `MusicgenProcessor` can be used to pre-process an audio prompt that is used for audio continuation. In the
following example, we load an audio file using the 🤗 Datasets library, pre-process it using the processor class,
and then forward the inputs to the model for generation:

In [ ]:
from datasets import load_dataset

dataset = load_dataset("sanchit-gandhi/gtzan", split="train", streaming=True)
sample = next(iter(dataset))["audio"]

# take the first half of the audio sample
sample["array"] = sample["array"][: len(sample["array"]) // 2]

inputs = processor(
    audio=sample["array"],
    sampling_rate=sample["sampling_rate"],
    text=["add heavy drums"],
    padding=True,
    return_tensors="pt",
)

audio_values = model.generate(**inputs.to(device), do_sample=True, guidance_scale=3, max_new_tokens=256)

Audio(audio_values[0].cpu().numpy(), rate=sampling_rate)

# Conclusion

In this notebook, we've seen how to:
1. Use **Coqui TTS** to generate speech from text.
2. Prepare for **Coqui TTS** training.
3. Generate short **music clips** using **MusicGen**.
